<a href="https://colab.research.google.com/github/mrdantzig/nlp-bible-code/blob/master/22%EC%9E%A5_%EC%8B%AC%EC%B8%B5%ED%95%99%EC%8A%B5%EC%9D%84_%EC%9D%B4%EC%9A%A9%ED%95%9C_%EC%9D%98%EB%AF%B8%EB%B6%84%EC%84%9D/%5B22-1%5D%20ELMO_Visualization%EC%9D%84%20%ED%86%B5%ED%95%9C%20Contextual%20Representation%EC%9D%98%20%EC%A4%91%EC%9D%98%EC%84%B1%20%ED%95%B4%EC%86%8C%20%ED%8C%8C%EC%95%85_%EC%88%98%EC%A0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [22-1] ELMo Visualization을 통한 Contextual Representation의 중의성 해소 파악 (수정본)

자연어처리 바이블 22장 실습 [22-1]을 최신 Colab에서 실행되도록 고친 노트북입니다. 같은 단어(plant, bank, work)라도 **문맥에 따라 ELMo 벡터가 달라지는지**를 PCA 2차원 산점도로 확인합니다(교재 22.2.1 Contextual Representation).

원본 출처: https://github.com/hengluchang/visualizing_contextual_vectors

**원본 대비 수정 사항**
1. **allennlp → TensorFlow Hub ELMo**: 원본의 `allennlp.commands.elmo.ElmoEmbedder`는 allennlp 1.0에서 삭제되었고, allennlp는 2022년에 개발이 끝나 최신 Colab(Python 3.12)에 설치되지 않습니다. 같은 원 논문 ELMo 모델(1B Word Benchmark 학습)을 TF Hub에서 불러옵니다. 층 번호는 원본과 같습니다(0: 문자 CNN, 1·2: biLSTM 1·2층, 1024차원).
2. **`pip install sklearn` 삭제**: `sklearn` 패키지 이름이 막혀 설치 오류가 납니다. scikit-learn·matplotlib·numpy는 Colab에 기본 설치되어 있습니다. 대신 `tensorflow_hub` import에 필요한 `setuptools<81`을 설치합니다.
3. **그림 바로 보기**: 원본은 png 파일로만 저장했지만, 노트북 안에 바로 표시하고 파일로도 저장합니다.
4. **bank·work 예제 실행 가능하게 정리**: 원본에서 주석 처리되어 있던 두 예제를 6번 셀에서 바로 돌려볼 수 있습니다.

※ 모델 파일(약 350MB)을 처음 한 번 내려받습니다. GPU는 필요 없습니다.

## 1. 환경 설정

In [ ]:
# tensorflow_hub가 쓰는 pkg_resources는 setuptools 81부터 빠져 있어 버전을 맞춥니다
!pip install -q -U tensorflow-hub "setuptools<81"

from collections import OrderedDict

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_hub as hub          # 수정: allennlp 대신 TF Hub
from sklearn.decomposition import PCA

print("TensorFlow", tf.__version__, "/ TF Hub", hub.__version__)

## 2. ELMo 클래스 선언

원본과 같은 `get_elmo_vector(tokens, layer)` 인터페이스를 유지합니다.

| layer | 원본(allennlp) | TF Hub 출력 키 | 차원 |
|---|---|---|---|
| 0 | 문자 CNN 임베딩 | `word_emb` | 512 |
| 1 | biLSTM 1층 | `lstm_outputs1` | 1024 |
| 2 | biLSTM 2층 | `lstm_outputs2` | 1024 |

In [ ]:
class Elmo:
    LAYER_KEYS = {0: 'word_emb', 1: 'lstm_outputs1', 2: 'lstm_outputs2'}

    def __init__(self, url="https://tfhub.dev/google/elmo/3"):
        # 수정: ElmoEmbedder() → TF Hub ELMo v3 (원 논문 모델)
        self.elmo = hub.load(url).signatures["default"]
        self.input_name = list(self.elmo.structured_input_signature[1].keys())[0]

    # ELMO 벡터 반환 함수.
    # 입력 token들과 layer에 따라 반환
    def get_elmo_vector(self, tokens, layer):
        out = self.elmo(**{self.input_name: tf.constant([" ".join(tokens)])})
        X = out[self.LAYER_KEYS[layer]][0, :len(tokens)].numpy()   # (토큰 수, 1024)
        return X


model = Elmo()
print(model.get_elmo_vector("I like this plant".split(), layer=1).shape)

## 3. 차원 축소 · 그림 그리기 함수 (원본과 동일, 그림 표시만 추가)

In [ ]:
def dim_reduction(X, n):
    pca = PCA(n_components=n)
    print("size of X: {}".format(X.shape))
    results = pca.fit_transform(X)
    print("size of reduced X: {}".format(results.shape))

    for i, ratio in enumerate(pca.explained_variance_ratio_):
        print("Variance retained ratio of PCA-{}: {}".format(i+1, ratio))

    return results


def pick_color(i):
    return ['ro', 'bo', 'yo', 'go', 'co'][i] if i < 4 else 'co'


def plot(word, token_list, reduced_X, file_name, title):
    fig, ax = plt.subplots(figsize=(8, 5))
    forms = [word, word + 's', word + 'ing', word + 'ed']

    # plot ELMo vectors
    i = 0
    for j, token in enumerate(token_list):
        color = pick_color(j)
        for _, w in enumerate(token):
            # only plot the word of interest
            if w.lower() in forms:
                ax.plot(reduced_X[i, 0], reduced_X[i, 1], color)
            i += 1

    tokens = []
    for token in token_list:
        tokens += token

    # annotate point
    k = 0
    for i, token in enumerate(tokens):
        if token.lower() in forms:
            text = ' '.join(token_list[k])
            # bold the word of interest in the sentence
            text = text.replace(token, r"$\bf{" + token + "}$")
            ax.annotate(text, xy=(reduced_X[i, 0], reduced_X[i, 1]))
            k += 1

    ax.set_title(title)
    ax.set_xlabel("PCA 1")
    ax.set_ylabel("PCA 2")
    fig.savefig(file_name, bbox_inches="tight")
    print("{} saved\n".format(file_name))
    plt.show()                                   # 수정: 노트북에 바로 표시


def visualize(word, sentences, layers=(1, 2)):
    token_list = [sentence.split() for sentence in sentences.values()]
    # Elmo의 Layer마다의 출력 결과 비교.
    for layer in layers:
        print("visualizing word {} using ELMo layer {}".format(word, layer))
        X = np.concatenate([model.get_elmo_vector(tokens=tokens, layer=layer)
                            for tokens in token_list], axis=0)
        # 2차원으로 축소
        X_reduce = dim_reduction(X=X, n=2)
        file_name = "{}_elmo_layer_{}.png".format(word, layer)
        title = "Layer {} ELMo vectors of the word {}".format(layer, word)
        plot(word, token_list, X_reduce, file_name, title)

## 4. Main — "plant"(심다 / 식물)

빨강·파랑·노랑 문장은 **동사 "심다"**, 초록·하늘색 문장은 **명사 "식물"**입니다.

In [ ]:
# Plants에 관련 문장 선언.
plants = OrderedDict()
plants[0] = "The gardener planted some trees in my yard"                     # 심는다의 의미.
plants[1] = "I plan to plant a Joshua tree tomorrow"                          # 심는다의 의미.
plants[2] = "My sister planted a seed and hopes it will grow to a tree"      # 심는다의 의미.
plants[3] = "This kind of plant only grows in the subtropical region"        # 식물의 의미.
plants[4] = "Most of the plants will die without water"                      # 식물의 의미.

visualize("plant", plants)

## 5. 결과 해석

- PCA는 47개 토큰 전체의 벡터로 학습하고, 그림에는 plant 계열 단어만 표시합니다.
- 같은 철자 "plant"라도 **문맥에 따라 다른 벡터**를 가지며, "심다" 문장과 "식물" 문장이 서로 다른 영역에 모입니다. 단어마다 벡터가 하나뿐인 Word2Vec·GloVe와의 차이입니다.
- 2층(layer 2)이 의미 정보를, 1층이 문법 정보를 더 많이 담는다고 알려져 있습니다(ELMo 논문). 두 그림을 비교해 보세요.

## 6. (추가) bank(은행 / 강둑), work(작품 / 일하다)도 확인하기

In [ ]:
# Bank에 대한 문장 선언.
banks = OrderedDict()
banks[0] = "One can deposit money at the bank"            # 은행관련 문장
banks[1] = "He had a nice walk along the river bank"      # 강둑관련 문장
banks[2] = "I withdrew cash from the bank"                # 은행관련 문장
banks[3] = "The river bank was not clean"                 # 강둑관련 문장
banks[4] = "My wife and I have a joint bank account"      # 은행관련 문장

# Work에 대한 문장 선언.
works = OrderedDict()
works[0] = "I like this beautiful work by Andy Warhol"                         # 명사로써 이해
works[1] = "Employee works hard every day"                                     # 동사로써 이해
works[2] = "My sister works at Starbucks"                                      # 동사로써 이해
works[3] = "This amazing work was done in the early nineteenth century"        # 명사로써 이해
works[4] = "Hundreds of people work in this building"                          # 동사로써 이해

visualize("bank", banks, layers=[2])
visualize("work", works, layers=[2])